# Block 0 — SpecMQuant reproduction gate

Runs the Block-0 correctness gate (IMPLEMENTATION_PLAN.md Phase 1): reproduce the
**direction** of SpecMQuant's finding — EAGLE speculative decoding's relative speedup
erodes when the target model is W4A16-quantized — on Llama-3-8B-Instruct, single-stream,
greedy, in vLLM.

**What it runs:** 8 cells = {FP16, W4A16-GPTQ-g128} x {no-spec, EAGLE} x {GSM8K, HumanEval},
4 server launches (one per model x spec combination), all driven by the repo harness.

**Requirements**
- **A100 runtime.** Runtime > Change runtime type > A100 GPU. Verify with the nvidia-smi
  cell below — don't trust the dropdown (PREREQ_RESULTS.md Check 1).
- **HF token** with access to `meta-llama/Meta-Llama-3-8B-Instruct` (gated: accept the
  license on the model page first). Store it as a Colab secret named `HF_TOKEN`
  (key icon in the left sidebar).

**Install recipe:** vLLM goes into an isolated `virtualenv` and runs as a subprocess —
NOT a bare pip install into the notebook kernel. Colab's preinstalled RAPIDS/torch stack
breaks a kernel-level vLLM install in confusing ways (PREREQ_RESULTS.md Check 6 recipe).

**Budget:** ~2–2.5 A100-hours end to end (two model downloads, 4 server startups at
~3 min each — Check 6 measured 172 s — plus ~40–70 min of timed measurement). **Note your
compute-units meter before and after: this run doubles as the Check-1 burn-rate calibration.**

In [2]:
# 1. Verify the GPU actually assigned (don't trust the runtime-type dropdown)
!nvidia-smi
import subprocess
gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu)
assert "A100" in gpu, "Not an A100 - reconnect until Colab honors the selection."
units_before = input("Compute-units balance right now (from the Colab usage panel): ")

Tue Jul  7 08:10:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# 2. Get the repo
%cd /content
!git clone https://github.com/manojarulmurugan/SpecDecoding-Study-vLLM-SGLang.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo

/content
Already up to date.
/content/repo


In [4]:
# 3. Isolated vLLM environment (Check 6 recipe: virtualenv, not stdlib venv;
#    ninja inside it for FlashInfer's JIT; ~6-8 min).
#    Pin: vllm==0.24.0 — confirmed to run this project's full stack (Check 6).
%pip install -q virtualenv
!virtualenv -q /content/vllm_env
!/content/vllm_env/bin/pip install -q vllm==0.24.0 datasets pyyaml requests pytest ninja
!/content/vllm_env/bin/python -c "import vllm; print('vllm', vllm.__version__)"

vllm 0.24.0


In [9]:
# 4. HF auth (gated Llama-3 weights)
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token set:", bool(os.environ["HF_TOKEN"]))

token set: True


In [10]:
# 5. Harness self-check inside the venv (no GPU needed, ~1 min).
#    If anything fails here, fix before burning GPU time.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m pytest tests -q

........................................................................ [ 98%]
.                                                                        [100%]
73 passed in 5.92s


In [11]:
# 6. Sanity: print the exact server commands the sweep will use (nothing launches)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/repro/repro_*.yaml" --results-dir results --dry-run

[sweep] 8 run(s) in 4 server group(s)
[sweep] group 1/4: 2 pending run(s)
[sweep] server command: vllm serve meta-llama/Meta-Llama-3-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.8 --max-model-len 4096 --seed 1234 --no-enable-prefix-caching --speculative-config {"draft_tensor_parallel_size": 1, "method": "eagle", "model": "yuhuili/EAGLE-LLaMA3-Instruct-8B", "num_speculative_tokens": 5}
[sweep]   would run: repro_meta-llama-3-8b-instruct_fp16-fp16-eagle_gsm8k_c1_r0
[sweep]   would run: repro_meta-llama-3-8b-instruct_fp16-fp16-eagle_humaneval_c1_r0
[sweep] group 2/4: 2 pending run(s)
[sweep] server command: vllm serve meta-llama/Meta-Llama-3-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.8 --max-model-len 4096 --seed 1234 --no-enable-prefix-caching
[sweep]   would run: repro_meta-llama-3-8b-instruct_fp16-fp16-none_gsm8k_c1_r0
[sweep]   would run: repro_meta-llama-3-8b-instruct_fp16-fp16-none_humaneval_c1_r0
[sweep

In [12]:
# 7. THE SWEEP. Resumable: if Colab disconnects, re-run this cell — completed
# cells are skipped and only the interrupted one re-runs (a disconnect costs one cell).
# The harness launches `vllm serve` as a subprocess; PATH points into the venv so it
# finds the right vllm binary and ninja (FlashInfer JIT). Server logs: results/server_logs/.
# Rough timing: FP16 groups ~35-50 min, W4A16 groups ~20-35 min.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/repro/repro_*.yaml" --results-dir results

[sweep] 8 run(s) in 4 server group(s)
[sweep] group 1/4: 2 pending run(s)
[sweep] server command: vllm serve meta-llama/Meta-Llama-3-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.8 --max-model-len 4096 --seed 1234 --no-enable-prefix-caching --speculative-config {"draft_tensor_parallel_size": 1, "method": "eagle", "model": "yuhuili/EAGLE-LLaMA3-Instruct-8B", "num_speculative_tokens": 5}
[sweep] waiting for server (log: results/server_logs/server_20260707_081401.log)





[run] repro_meta-llama-3-8b-instruct_fp16-fp16-eagle_gsm8k_c1_r0: 100 prompts, concurrency=1, max_new_tokens=256
  [load] 25/100 requests done
  [load] 50/100 requests done
  [load] 75/100 requests done
  [load] 100/100 requests done
[run] repro_meta-llama-3-8b-instruct_fp16-fp16-eagle_gsm8k_c1_r0 -> results/runs/repro_meta-llama-3-8b-instruct_fp16-fp16-eagle_gsm8k_c1_r0.json (status=ok, 148.6 tok/s mean/request, tau=2.486, acc=0.780)



[run] repro_meta-llama-3-8b-instruct_fp16-fp1

In [13]:
!grep -n -iE "error|exception|traceback|assert|marlin|gptq|compressed|out of memory|cuda" results/server_logs/server_20260707_083855.log | head -60

13:(APIServer pid=25531) ERROR 07-07 08:39:09 [repo_utils.py:68] Error retrieving safetensors: 'YudiZh/Meta-Llama-3-8B-Instruct-W4A16-g128' is not a safetensors repo. Couldn't find 'model.safetensors.index.json' or 'model.safetensors' files., retrying 1 of 2
14:(APIServer pid=25531) ERROR 07-07 08:39:11 [repo_utils.py:66] Error retrieving safetensors: 'YudiZh/Meta-Llama-3-8B-Instruct-W4A16-g128' is not a safetensors repo. Couldn't find 'model.safetensors.index.json' or 'model.safetensors' files.
18:(EngineCore pid=25692) INFO 07-07 08:39:28 [core.py:114] Initializing a V1 LLM engine (v0.24.0) with config: model='YudiZh/Meta-Llama-3-8B-Instruct-W4A16-g128', speculative_config=SpeculativeConfig(method='eagle', model='yuhuili/EAGLE-LLaMA3-Instruct-8B', num_spec_tokens=5), tokenizer='YudiZh/Meta-Llama-3-8B-Instruct-W4A16-g128', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir

In [14]:
import os
from huggingface_hub import snapshot_download
from safetensors.torch import load_file, save_file
import shutil

src = snapshot_download("YudiZh/Meta-Llama-3-8B-Instruct-W4A16-g128", token=os.environ["HF_TOKEN"])
dst = "/content/llama3-8b-w4a16-g128-fixed"
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree(src, dst)

weights_path = os.path.join(src, "model_gptq.safetensors")
tensors = load_file(weights_path)
os.remove(os.path.join(dst, "model_gptq.safetensors"))

fixed = {}
for k, v in tensors.items():
    if k.startswith("lm_head.") or k.startswith("model."):
        fixed[k] = v
    else:
        fixed["model." + k] = v

save_file(fixed, os.path.join(dst, "model.safetensors"), metadata={"format": "pt"})
print("keys before:", list(tensors.keys())[:5])
print("keys after:", list(fixed.keys())[:5])
print("saved to", dst)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

keys before: ['model.layers.0.mlp.down_proj.qweight', 'model.layers.0.mlp.gate_up_proj.qweight', 'model.layers.0.self_attn.o_proj.qweight', 'model.layers.0.self_attn.qkv_proj.qweight', 'model.layers.1.mlp.down_proj.qweight']
keys after: ['model.layers.0.mlp.down_proj.qweight', 'model.layers.0.mlp.gate_up_proj.qweight', 'model.layers.0.self_attn.o_proj.qweight', 'model.layers.0.self_attn.qkv_proj.qweight', 'model.layers.1.mlp.down_proj.qweight']
saved to /content/llama3-8b-w4a16-g128-fixed


In [15]:
!grep -n -iE "marlin|desc_act|falling back|quant_method|gptq|not supported" results/server_logs/server_20260707_083855.log | head -40

18:(EngineCore pid=25692) INFO 07-07 08:39:28 [core.py:114] Initializing a V1 LLM engine (v0.24.0) with config: model='YudiZh/Meta-Llama-3-8B-Instruct-W4A16-g128', speculative_config=SpeculativeConfig(method='eagle', model='yuhuili/EAGLE-LLaMA3-Instruct-8B', num_spec_tokens=5), tokenizer='YudiZh/Meta-Llama-3-8B-Instruct-W4A16-g128', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=auto_gptq, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_par

In [16]:
import os
from huggingface_hub import snapshot_download
from safetensors.torch import load_file, save_file
import shutil

src = snapshot_download("YudiZh/Meta-Llama-3-8B-Instruct-W4A16-g128", token=os.environ["HF_TOKEN"])
dst = "/content/llama3-8b-w4a16-g128-fixed"
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree(src, dst)

weights_path = os.path.join(src, "model_gptq.safetensors")
tensors = load_file(weights_path)
os.remove(os.path.join(dst, "model_gptq.safetensors"))

fixed = {}
for k, v in tensors.items():
    if k.startswith("lm_head.") or k.startswith("model."):
        fixed[k] = v
    else:
        fixed["model." + k] = v

save_file(fixed, os.path.join(dst, "model.safetensors"), metadata={"format": "pt"})
print("saved to", dst)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

saved to /content/llama3-8b-w4a16-g128-fixed


In [17]:
import json, os

dst = "/content/llama3-8b-w4a16-g128-fixed"

with open(os.path.join(dst, "config.json")) as f:
    config = json.load(f)

with open(os.path.join(dst, "quantize_config.json")) as f:
    quant_cfg = json.load(f)

config["quantization_config"] = quant_cfg
with open(os.path.join(dst, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print("quantization_config now in config.json:", config["quantization_config"])

quantization_config now in config.json: {'bits': 4, 'group_size': 128, 'damp_percent': 0.01, 'desc_act': True, 'static_groups': True, 'sym': True, 'true_sequential': True, 'quant_method': 'gptq', 'checkpoint_format': 'gptq'}


In [18]:
!grep -n "model:" configs/repro/repro_w4a16_*.yaml

configs/repro/repro_w4a16_eagle_gsm8k.yaml:6:model: YudiZh/Meta-Llama-3-8B-Instruct-W4A16-g128
configs/repro/repro_w4a16_eagle_gsm8k.yaml:11:draft_model: yuhuili/EAGLE-LLaMA3-Instruct-8B
configs/repro/repro_w4a16_eagle_humaneval.yaml:6:model: YudiZh/Meta-Llama-3-8B-Instruct-W4A16-g128
configs/repro/repro_w4a16_eagle_humaneval.yaml:11:draft_model: yuhuili/EAGLE-LLaMA3-Instruct-8B
configs/repro/repro_w4a16_eagle_humaneval.yaml:15:  tokenizer_model: YudiZh/Meta-Llama-3-8B-Instruct-W4A16-g128
configs/repro/repro_w4a16_none_gsm8k.yaml:6:model: YudiZh/Meta-Llama-3-8B-Instruct-W4A16-g128
configs/repro/repro_w4a16_none_humaneval.yaml:6:model: YudiZh/Meta-Llama-3-8B-Instruct-W4A16-g128
configs/repro/repro_w4a16_none_humaneval.yaml:14:  tokenizer_model: YudiZh/Meta-Llama-3-8B-Instruct-W4A16-g128


In [19]:
!sed -i 's|model: YudiZh/Meta-Llama-3-8B-Instruct-W4A16-g128|model: /content/llama3-8b-w4a16-g128-fixed|' \
  configs/repro/repro_w4a16_none_humaneval.yaml \
  configs/repro/repro_w4a16_none_gsm8k.yaml \
  configs/repro/repro_w4a16_eagle_gsm8k.yaml \
  configs/repro/repro_w4a16_eagle_humaneval.yaml
!grep -n "model:" configs/repro/repro_w4a16_*.yaml

configs/repro/repro_w4a16_eagle_gsm8k.yaml:6:model: /content/llama3-8b-w4a16-g128-fixed
configs/repro/repro_w4a16_eagle_gsm8k.yaml:11:draft_model: yuhuili/EAGLE-LLaMA3-Instruct-8B
configs/repro/repro_w4a16_eagle_humaneval.yaml:6:model: /content/llama3-8b-w4a16-g128-fixed
configs/repro/repro_w4a16_eagle_humaneval.yaml:11:draft_model: yuhuili/EAGLE-LLaMA3-Instruct-8B
configs/repro/repro_w4a16_eagle_humaneval.yaml:15:  tokenizer_model: /content/llama3-8b-w4a16-g128-fixed
configs/repro/repro_w4a16_none_gsm8k.yaml:6:model: /content/llama3-8b-w4a16-g128-fixed
configs/repro/repro_w4a16_none_humaneval.yaml:6:model: /content/llama3-8b-w4a16-g128-fixed
configs/repro/repro_w4a16_none_humaneval.yaml:14:  tokenizer_model: /content/llama3-8b-w4a16-g128-fixed


In [20]:
!ls -la /content/llama3-8b-w4a16-g128-fixed/model.safetensors

-rw------- 1 root root 5700628912 Jul  7 08:57 /content/llama3-8b-w4a16-g128-fixed/model.safetensors


In [21]:
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/repro/repro_*.yaml" --results-dir results

[sweep] 8 run(s) in 4 server group(s)
[sweep] skip (complete): repro_meta-llama-3-8b-instruct_fp16-fp16-eagle_gsm8k_c1_r0
[sweep] skip (complete): repro_meta-llama-3-8b-instruct_fp16-fp16-eagle_humaneval_c1_r0
[sweep] skip (complete): repro_meta-llama-3-8b-instruct_fp16-fp16-none_gsm8k_c1_r0
[sweep] skip (complete): repro_meta-llama-3-8b-instruct_fp16-fp16-none_humaneval_c1_r0
[sweep] group 3/4: 2 pending run(s)
[sweep] server command: vllm serve /content/llama3-8b-w4a16-g128-fixed --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.8 --max-model-len 4096 --seed 1234 --no-enable-prefix-caching --speculative-config {"draft_tensor_parallel_size": 1, "method": "eagle", "model": "yuhuili/EAGLE-LLaMA3-Instruct-8B", "num_speculative_tokens": 5}
[sweep] waiting for server (log: results/server_logs/server_20260707_090015.log)
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/c

In [22]:
!grep -n -iE "error|exception|keyerror|valueerror|traceback|assert|marlin|desc_act|out of memory|cuda" results/server_logs/server_20260707_090015.log | head -60

16:(EngineCore pid=31358) INFO 07-07 09:00:40 [core.py:114] Initializing a V1 LLM engine (v0.24.0) with config: model='/content/llama3-8b-w4a16-g128-fixed', speculative_config=SpeculativeConfig(method='eagle', model='yuhuili/EAGLE-LLaMA3-Instruct-8B', num_spec_tokens=5), tokenizer='/content/llama3-8b-w4a16-g128-fixed', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=auto_gptq, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='',

In [23]:
!tail -n 100 results/server_logs/server_20260707_090015.log

(EngineCore pid=31358)     return func(*args, **kwargs)
(EngineCore pid=31358)            ^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=31358)   File "/content/vllm_env/lib/python3.12/site-packages/vllm/model_executor/model_loader/base_loader.py", line 64, in load_model
(EngineCore pid=31358)     self.load_weights(model, model_config)
(EngineCore pid=31358)   File "/content/vllm_env/lib/python3.12/site-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
(EngineCore pid=31358)     return func(*args, **kwargs)
(EngineCore pid=31358)            ^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=31358)   File "/content/vllm_env/lib/python3.12/site-packages/vllm/model_executor/model_loader/default_loader.py", line 427, in load_weights
(EngineCore pid=31358)     loaded_weights = model.load_weights(self.get_all_weights(model_config, model))
(EngineCore pid=31358)                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=31358)   File "/content/vllm_env/lib/python3.12

In [ ]:
# 8. Gate verdict
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m analysis.repro_gate results --targets configs/repro/reference_targets.yaml

In [ ]:
# 9. Preserve everything (results/ is git-ignored by design — archive it)
units_after = input("Compute-units balance now: ")
print("Units burned by this session:", units_before, "->", units_after)
!zip -qr block0_results.zip results
from google.colab import files
files.download("block0_results.zip")

## Reading the result

- **GATE: PASS** — direction reproduced. Archive `results/`, record the burn rate in
  PREREQ_RESULTS.md Check 1, proceed to Phase 2.
- **GATE: FAIL** — stop (PROJECT_SPEC.md §7.1). Debug order: (1) read
  `results/server_logs/*.log` for the failing group — EAGLE misconfig fails loudly at
  launch; (2) check tau in the report — tau ≈ 1 means the draft head isn't accepting
  anything (wrong draft/target pairing); (3) check the accuracy columns — greedy spec
  decode is output-preserving, so accuracy drift between spec-on/off cells means a
  harness bug, not a model property.
- **Magnitude warnings are expected, failures are not.** SpecMQuant's engine is bespoke
  C/CUDA with tree drafting (size 60); vLLM EAGLE is chain drafting
  (`num_speculative_tokens: 5`), and our W4A16 checkpoint is their non-rotation variant.
  Absolute speedups will differ; document the gap and its cause (EXPERIMENT_MATRIX.md §7,
  revised tolerance).

Known-good references (Figure 1a of arXiv 2505.22179, their engine, A100):
EAGLE-2 rel speedup 2.3x on FP16 -> 1.3x on W4A16; W4A16 quant-only 2.1x.
Provenance + gate thresholds: `configs/repro/reference_targets.yaml`.